# Seasonal climatology RMSE in the CMIP6 context — alternate spin-ups vs Full-CPL

How far the spun-up atmosphere is from observations compared with the CMIP6 models: the e3sm_diags seasonal
climatology RMSE of nine fields (net TOA radiation, SW/LW cloud radiative effect, precipitation, land surface air
temperature, sea-level pressure, 200/850 hPa zonal wind, 500 hPa geopotential height; CERES-EBAF 4.1, GPCP 2.3,
ERA5) of each run, against the spread of the CMIP6 historical models from the e3sm_diags CMIP6 benchmark
(`data/cmip6/cmip6_historical_seasonal_rmse_202203.csv`, 52 models, plus E3SM-2-0).

Figures:
* `cmip6_rmse_boxes_<window>.pdf` — one panel per field: CMIP6 box (25–75 %, whiskers min–max, line median, dot mean)
  per season, with FC, FC-FOSI-S1/S2 and E3SM-2-0 over `WINDOWS`
* `cmip6_rmse_ANN_evolution.pdf` — annual RMSE of every `EVOLUTION_LEN`-yr e3sm_diags window of each run against the
  CMIP6 interquartile range (Full-CPL piControl windows continue after the spin-up)

Note: the CMIP6 values are historical simulations (1985–2014 climatologies) while the runs are pre-industrial
control states; the comparison places the spin-up errors in the multi-model spread, it is not like-for-like.

Inputs: the e3sm_diags `<season>_metrics_table.csv` files of each run (`data/diag_data/`), the CMIP6 table
(`data/cmip6/`). Figures go to `FIG_DIR` (public_html), extracted RMSE tables to `DATA_DIR`.

In [ ]:
import os, sys
WF_ROOT = os.path.abspath("..")            # coupled_spinup_eval/
if WF_ROOT not in sys.path:
    sys.path.insert(0, WF_ROOT)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
%matplotlib inline

import matplotlib.pyplot as plt

import glob, re
import numpy as np
import xarray as xr

from util.experiments import make_run_dict
from util.figures import publish
from util.cmip6_rmse import FIELDS, SEASONS, read_cmip6_csv, read_e3sm_diags_rmse, plot_rmse_boxes

## Setup

In [ ]:
# ---- paths ----
MODEL_ROOT   = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/E3SMv3_testings"     # case directories
OUTPUT_ROOT  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3spinup_paper"   # published figures/ and data/
OUTPUT_URL   = "https://web.lcrc.anl.gov/public/e3sm/diagnostic_output/ac.szhang/v3spinup_paper"
FIG_DIR      = f"{OUTPUT_ROOT}/figures/cmip6_rmse"
DATA_DIR     = f"{OUTPUT_ROOT}/data/cmip6_rmse"            # cached/derived data (index: data/README.md)

DIAG_DATA    = f"{OUTPUT_ROOT}/data/diag_data"             # e3sm_diags tables: <case>/e3sm_diags/atm_monthly_180x360_aave/model_vs_obs_<yrs>/viewer/table-data/
CMIP6_CSV    = f"{OUTPUT_ROOT}/data/cmip6/cmip6_historical_seasonal_rmse_202203.csv"

# ---- experiments: case directory names (under MODEL_ROOT) and model-year periods ----
EXPERIMENTS = {
    "Full-CPL": {
        "spin-up":   {"name": "20231209.v3.LR.piControl-spinup.chrysalis",      "period": "0001-2000"},
        "piControl": {"name": "v3.LR.piControl",                                "period": "0001-0500"},
    },
    "AltBG1": {
        "spin-up":   {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240707.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
    "AltBG2": {
        "spin-up":   {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0001-0220"},
        "piControl": {"name": "20240818.v3.LR.piControl.Rerun.GC2BC.chrysalis", "period": "0221-0350"},
    },
}
LABEL = {"Full-CPL": "FC", "AltBG1": "FC-FOSI-S1", "AltBG2": "FC-FOSI-S2", "AltBG3": "FC-FOSI-S3"}

# ---- analysis settings ----
EXPS      = ["Full-CPL", "AltBG1", "AltBG2"]   # runs shown (spin-up cases)
COLORS    = ["black", "#cc0000", "#377eb8"]    # one per run
WINDOWS   = ["0301-0350"]   # e3sm_diags model_vs_obs windows, one box figure each
REFERENCE = "E3SM-2-0"      # CMIP6-table row drawn as an extra marker (None: none)
EVOLUTION_LEN = 50          # window length (yr) of the ANN evolution figure
CTRL_OFFSET   = 2000        # piControl windows of Full-CPL are drawn after the spin-up (year + CTRL_OFFSET)

In [ ]:
# Shared inputs/outputs
run_dict = make_run_dict(EXPERIMENTS, MODEL_ROOT)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
publish(OUTPUT_ROOT, FIG_DIR)
print("figures ->", FIG_DIR)
print("data    ->", DATA_DIR)
print("web     ->", FIG_DIR.replace(OUTPUT_ROOT, OUTPUT_URL))

CASES  = [run_dict[t]["spin-up"]["name"] for t in EXPS]
LABELS = [LABEL[t] for t in EXPS]

def table_dir(case, window):
    return f"{DIAG_DATA}/{case}/e3sm_diags/atm_monthly_180x360_aave/model_vs_obs_{window}/viewer/table-data"

cmip6, e3sm2 = read_cmip6_csv(CMIP6_CSV)
print(f"CMIP6: {cmip6.sizes['model']} models; E3SM rows: {list(e3sm2['model'].values)}")

## 1 RMSE per field and season against the CMIP6 spread
### Settings

In [ ]:
BOX_STYLE = dict(ncols=3, fontz=12.0, panel_letters=True)
MARKERS   = {"run": "^", "reference": "s"}
REF_COLOR = "#4daf4a"

### Compute & plot

In [ ]:
for window in WINDOWS:
    runs = {}
    for case, label, color in zip(CASES, LABELS, COLORS):
        r = read_e3sm_diags_rmse(table_dir(case, window))
        if r.isnull().all():
            print(f"[skip] {label}: no e3sm_diags tables for {window}")
            continue
        runs[label] = dict(rmse=r, color=color, marker=MARKERS["run"])
    ref = None if REFERENCE is None else {REFERENCE: dict(rmse=e3sm2.sel(model=REFERENCE), color=REF_COLOR,
                                                          marker=MARKERS["reference"])}
    xr.Dataset({lab: r["rmse"] for lab, r in runs.items()}).to_netcdf(os.path.join(DATA_DIR, f"rmse_{window}.nc"))
    out = os.path.join(FIG_DIR, f"cmip6_rmse_boxes_{window}.pdf")
    fig, axes = plot_rmse_boxes(cmip6, runs, reference=ref, **BOX_STYLE, save=out)
    plt.show()
    print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))

publish(FIG_DIR, DATA_DIR)

## 2 Annual RMSE through the spin-up
Every `EVOLUTION_LEN`-yr e3sm_diags window of each run (and of the Full-CPL piControl, shifted by `CTRL_OFFSET`),
grey band = CMIP6 interquartile range, dashed = CMIP6 median.

In [ ]:
def windows_of(case, length):
    out = []
    for d in glob.glob(f"{DIAG_DATA}/{case}/e3sm_diags/atm_monthly_180x360_aave/model_vs_obs_*"):
        m = re.search(r"model_vs_obs_(\d{4})-(\d{4})$", d)
        if m and int(m[2]) - int(m[1]) + 1 == length:
            out.append((int(m[1]), int(m[2])))
    return sorted(out)

series = {}
for t, case, label, color in zip(EXPS, CASES, LABELS, COLORS):
    parts = [(case, 0)] + ([(run_dict[t]["piControl"]["name"], CTRL_OFFSET)]
                           if run_dict[t]["piControl"]["name"] != case else [])
    xs, ys = [], []
    for c, off in parts:
        for y0, y1 in windows_of(c, EVOLUTION_LEN):
            xs.append(off + (y0 + y1) / 2)
            ys.append(read_e3sm_diags_rmse(table_dir(c, f"{y0:04d}-{y1:04d}")).sel(season="ANN"))
    if xs:
        series[label] = (np.array(xs), xr.concat(ys, "window"), color)
        print(f"{label}: {len(xs)} windows")

names = list(FIELDS)
fig, axes = plt.subplots(3, 3, figsize=(16, 9), sharex=True, layout="constrained")
for k, (ax, f) in enumerate(zip(axes.flat, names)):
    q = cmip6.sel(field=f, season="ANN").quantile([0.25, 0.5, 0.75], "model").values
    ax.axhspan(q[0], q[2], color="0.88", zorder=0)
    ax.axhline(q[1], color="0.5", ls="--", lw=1)
    for label, (x, y, color) in series.items():
        ax.plot(x, y.sel(field=f), "o-", color=color, ms=3, lw=1.2, label=label)
    if CTRL_OFFSET:
        ax.axvline(CTRL_OFFSET, color="0.4", ls=":", lw=1)
    _, unit, title = FIELDS[f]
    ax.set_title(f"({'abcdefghi'[k]}) {title} ({unit})", fontsize=11, loc="left")
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("Model year (window centre)")
h, l = axes.flat[0].get_legend_handles_labels()
fig.legend(h, l + [], loc="outside lower center", ncol=len(l), frameon=False)
out = os.path.join(FIG_DIR, "cmip6_rmse_ANN_evolution.pdf")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()

publish(FIG_DIR, DATA_DIR)
print("web:", out.replace(OUTPUT_ROOT, OUTPUT_URL))